# Steps 3 + 4 on the real clips

Needs `colab_setup.ipynb` run first in this Colab session (same runtime).

1. Step 3: speech analysis of each clip (who spoke when, words with timings, prosody, emotion).
2. A rough check of step 3 against the clips' Gemini labels (not verified, so only indicative).
3. Step 4: scene context + translation from step 3's real output (Hindi clips to Tamil, Tamil clips to Hindi).

Everything is also copied to Drive (`nlp-project-clips/outputs/`), so the Mac can use it.

In [18]:
# 1. Paths (and a check that setup ran in this session)
import os, json, shutil, subprocess
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/nlp-project-clips")
REPO = Path("/content/nlp-project")
PY = str(REPO / ".venv/bin/python")
ENV = {**os.environ, "HF_HOME": str(DRIVE / "hf-cache"), "PYTHONPATH": str(REPO)}
assert Path(PY).exists() and (REPO / ".env").exists(), "Run colab_setup.ipynb first in this runtime"
CLIPS = {"hi_01": ("hi", "ta"), "hi_02": ("hi", "ta"), "ta_01": ("ta", "hi"), "ta_02": ("ta", "hi")}
missing = [c for c in CLIPS if not (REPO / f"data/clips/{c}.mp4").exists()]
print("ready" if not missing else f"missing clips: {missing}")

ready


In [19]:
# 2. Step 3 on each clip (first run downloads the models to Drive: ~5-10 min; later runs are fast)
for clip, (src, _) in CLIPS.items():
    r = subprocess.run([PY, "-m", "backend.pipeline.speech.run", f"data/clips/{clip}.mp4", "--lang", src],
                       cwd=REPO, env=ENV, capture_output=True, text=True)
    print(f"{clip}: {r.stdout.strip().splitlines()[-1] if r.returncode == 0 else 'FAILED'}")
    if r.returncode != 0:
        print(r.stderr[-2000:])
out = DRIVE / "outputs" / "speech"
out.mkdir(parents=True, exist_ok=True)
for f in (REPO / "data/cache/speech").glob("*"):
    shutil.copy(f, out / f.name)
print("copied to Drive:", sorted(p.name for p in out.iterdir()))

hi_01: 3 lines, 2 speakers, 25.47 s audio in 49.49 s -> /content/nlp-project/data/cache/speech/hi_01.json
hi_02: 9 lines, 4 speakers, 35.75 s audio in 17.34 s -> /content/nlp-project/data/cache/speech/hi_02.json
ta_01: 7 lines, 2 speakers, 33.88 s audio in 18.67 s -> /content/nlp-project/data/cache/speech/ta_01.json
ta_02: 6 lines, 4 speakers, 33.39 s audio in 16.32 s -> /content/nlp-project/data/cache/speech/ta_02.json
copied to Drive: ['hi_01.json', 'hi_01.npz', 'hi_02.json', 'hi_02.npz', 'ta_01.json', 'ta_01.npz', 'ta_02.json', 'ta_02.npz']


In [20]:
# 3. What step 3 found, line by line
for clip in CLIPS:
    d = json.loads((REPO / f"data/cache/speech/{clip}.json").read_text())
    m = d["meta"]
    print(f"\n=== {clip}: {len(d['lines'])} lines, {len(m['speakers'])} speakers, music {m['music_db']} dB, "
          f"Demucs used: {m['demucs_used']}, {m['timings_s']['total']} s")
    for l in d["lines"]:
        e = l["emotion"]
        print(f"  {l['start']:5.1f}-{l['end']:5.1f} {l['speaker']}  e2v={e['arousal_e2v']:.2f} "
              f"({e['label']:8s}) | {l['text'][:60]}")
        print(f"{'':27s}{l['text_roman'][:70]}")


=== hi_01: 3 lines, 2 speakers, music -9.4 dB, Demucs used: True, 49.49 s
    0.2-  6.2 S1  e2v=1.00 (happy   ) | हा चलाइए चलाइए ना चला चला चलाइए चलाइए तो बेटा तुम दोनों कच्छ
                           ha chalaaie chalaaie na chala chala chalaaie chalaaie to beta tum dono
    5.7- 15.7 S2  e2v=0.96 (happy   ) | भ बबलू कौन है र हमे बबलू बबलू है तुम्ारे बाप का का नाम है जग
                           bha babloo kaun hai ra hame babloo babloo hai tumaare baap ka ka naam 
   18.3- 25.6 S2  e2v=0.82 (happy   ) | आधार कार्ड नहीं बेटा एक रहबट मम सुधार कार्ड बना देंगे तुम्हा
                           aadhaar kaard naheen beta ek rahbat mam sudhaar kaard bana denge tumha

=== hi_02: 9 lines, 4 speakers, music -13.3 dB, Demucs used: True, 17.34 s
    0.0-  5.9 S1  e2v=0.81 (happy   ) | साक्षात पामदेव चले आ रहे हैं  क्या क्या किस्से सुनाएंगे आ जा
                           saakshaat paamdev chale a rahe hain  kya kya kisse sunaaenge a ja beta
    7.7- 10.4 S1  e2v=1.00 (happy   ) | राम भाई रात क

In [21]:
%%bash
# 4. Rough check against the Gemini labels (unverified, so indicative only):
#    character error rate of the whole clip's text, and number of speakers found vs labelled
cd /content/nlp-project
.venv/bin/python - <<'EOF'
import json, re, unicodedata, jiwer
def norm(t):
    t = unicodedata.normalize("NFC", t)
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", t)).strip()
for clip in ["hi_01", "hi_02", "ta_01", "ta_02"]:
    ref_d = json.load(open(f"data/labels/{clip}.json"))
    hyp_d = json.load(open(f"data/cache/speech/{clip}.json"))
    ref = norm(" ".join(l["transcript"] for l in ref_d["lines"]))
    hyp = norm(" ".join(l["text"] for l in hyp_d["lines"]))
    print(f"{clip}: CER {jiwer.cer(ref, hyp):.1%}  WER {jiwer.wer(ref, hyp):.1%}  "
          f"speakers found {len(hyp_d['meta']['speakers'])} vs labelled {len(ref_d.get('speakers', []))}")
EOF

hi_01: CER 29.8%  WER 36.4%  speakers found 2 vs labelled 6
hi_02: CER 17.0%  WER 28.2%  speakers found 4 vs labelled 7
ta_01: CER 24.7%  WER 37.3%  speakers found 2 vs labelled 3
ta_02: CER 41.8%  WER 45.7%  speakers found 4 vs labelled 3


In [22]:
# 5. Step 4 on step 3's real output (uses Groq; each clip costs ~20-40k of the 200k daily tokens)
for clip, (src, tgt) in CLIPS.items():
    r = subprocess.run([PY, "scripts/dub_clip.py", f"data/clips/{clip}.mp4", "--speech", "auto", "--to", tgt],
                       cwd=REPO, env=ENV, capture_output=True, text=True)
    print(f"################ {clip}: {src} -> {tgt}")
    print(r.stdout[-6000:] if r.returncode == 0 else "FAILED\n" + r.stderr[-2000:])
out = DRIVE / "outputs" / "dub"
out.mkdir(parents=True, exist_ok=True)
for f in (REPO / "data/cache/dub").glob("*.json"):
    shutil.copy(f, out / f.name)
print("copied to Drive:", sorted(p.name for p in out.iterdir()))

KeyboardInterrupt: 

In [ ]:
# 6. Which model produced each translation (gpt-oss-120b is the one that counts for quality)
from collections import Counter
for clip, (src, tgt) in CLIPS.items():
    p = REPO / f"data/cache/dub/{clip}.{src}-{tgt}.json"
    if not p.exists():
        print(clip, "no output"); continue
    d = json.loads(p.read_text())
    print(clip, dict(Counter(l["dub"].get("_provider") for l in d["lines"])),
          "| context:", d["context"].get("_provider"), "| captions:", (d["captions"] or {}).get("_provider"))